<a href="https://colab.research.google.com/github/suzy-hur/IOD_lab_work/blob/main/email_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Motivation and Goals

This project aims to leverage Python's analytical capabilities to evaluate the quality of AI-generated email drafts compared to human-sent versions within a digital-only Australian bank specializing in home loans. The core motivation and goals are:

**Motivation:**
*   To demonstrate Python as a superior tool for accurate, deterministic, and reliable data analysis in quality checking AI-assisted versus lender-sent emails.
*   To avoid relying on another LLM for quality assessment, emphasising a programmatic, quantifiable approach.
*   To provide insights into the quality and effectiveness of AI-drafted communications concerning critical home loan assessment criteria and customer follow-ups (e.g., clarifying application data, income, expenses, discrepancies, debt reduction, loan term extensions, or duplicate applications).

**Goals:**
*   Quantify the accuracy, variance, and overall reliability of AI-generated email versions against human-sent emails.
*   Identify specific metrics and logical figures to assess and improve the quality and performance of the AI email drafting agent.
*   Develop a robust framework for continuous quality checking and enhancement of AI-assisted customer communications within the home loan application process.

In [29]:
import pandas as pd

# Load dataset into pandas dataframe, using the second row as the header
dataset = '/content/sample_data/Emails.xlsx'
df = pd.read_excel(dataset, header=1)

# Display DataFrame information and head to inspect the loaded data
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 27 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   101 non-null    object
 1   Unnamed: 1   101 non-null    object
 2   Unnamed: 2   101 non-null    object
 3   Unnamed: 3   101 non-null    object
 4   Unnamed: 4   101 non-null    object
 5   Unnamed: 5   101 non-null    object
 6   Unnamed: 6   101 non-null    object
 7   Unnamed: 7   101 non-null    object
 8   Unnamed: 8   101 non-null    object
 9   Unnamed: 9   101 non-null    object
 10  Unnamed: 10  101 non-null    object
 11  Unnamed: 11  101 non-null    object
 12  Unnamed: 12  101 non-null    object
 13  Unnamed: 13  101 non-null    object
 14  Unnamed: 14  101 non-null    object
 15  Unnamed: 15  101 non-null    object
 16  Unnamed: 16  101 non-null    object
 17  Unnamed: 17  101 non-null    object
 18  Unnamed: 18  101 non-null    object
 19  Unnamed: 19  101 non-null    

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,record_id,application_id,customer_name_masked,applicant_structure,customer_state,channel,loan_purpose,occupancy,application_stage,scenario_type,...,ai_subject,actual_subject,ai_points_requested,actual_points_requested,ai_draft_email,actual_human_email,human_edit_types,human_edit_summary,final_status,synthetic_internal_note
1,1,HL-2026-549245,Madison Scott / Archie Parker,Joint,WA,Direct digital,Refinance,Investment,Conditional approval,Self-employed income verification,...,Application review - please provide documents,Thanks for your application - a few items to c...,the last 2 years of Notices of Assessment | th...,the latest 2 years of personal and business No...,"Hi Madison and Archie,\n\nThank you for applyi...","Hi Madison and Archie,\n\nThanks for submittin...",removed a generic deadline; tightened the scop...,Human version is more specific about the evide...,Condition cleared,Self-employed income needs standard verificati...
2,2,HL-2025-575763,Max Smith,Single,ACT,Broker-introduced,Top-up,Owner Occupied,Conditional approval,Bonus/overtime verification,...,Home loan application - additional information...,Thanks for your application - a few items to c...,your last 3 payslips showing overtime or bonus...,your 2 most recent payslips showing the overti...,"Hi Max,\n\nThanks for your home loan applicati...","Hi Max,\n\nI've reviewed your application and ...",removed a generic deadline; reordered requests...,Human version improves clarity and sequencing....,Awaiting customer response,Variable income has been included in servicing...
3,3,HL-2026-172574,Alexander Nguyen,Single,NSW,Online refinance,Purchase,Investment,Verification,Dependants and family costs,...,Follow up on your application - dependants and...,Application follow up: dependants and expenses,confirmation of the number of dependants in yo...,confirmation of the number of dependants to be...,"Hi Alexander,\n\nThank you for applying with C...","Hi Alexander,\n\nThanks for your application. ...",made the request more customer-friendly; added...,Human version is more specific about the evide...,Duplicate application closed,Household composition and related costs do not...
4,4,HL-2025-801474,Thomas White,Single,NSW,Direct digital,Refinance,Investment,Verification,Outstanding bank statements,...,Application review - please provide documents,Application follow up: bank statements,the last 3 months of your main transaction acc...,the most recent statements for the main accoun...,"Hi Thomas,\n\nI am writing regarding your appl...","Hi Thomas,\n\nThanks for submitting your appli...",more specific document wording; reduced ambigu...,Human version is more specific about the evide...,Withdrawn by customer,Uploaded statements are incomplete or do not c...


In [30]:
df.columns = df.iloc[0]
df = df[1:]
df = df.reset_index(drop=True)

print(df.info())
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   record_id                100 non-null    object
 1   application_id           100 non-null    object
 2   customer_name_masked     100 non-null    object
 3   applicant_structure      100 non-null    object
 4   customer_state           100 non-null    object
 5   channel                  100 non-null    object
 6   loan_purpose             100 non-null    object
 7   occupancy                100 non-null    object
 8   application_stage        100 non-null    object
 9   scenario_type            100 non-null    object
 10  issue_tags               100 non-null    object
 11  lender_name              100 non-null    object
 12  lender_role              100 non-null    object
 13  ai_drafted_time_local    100 non-null    object
 14  human_sent_time_local    100 non-null    ob

,record_id,application_id,customer_name_masked,applicant_structure,customer_state,channel,loan_purpose,occupancy,application_stage,scenario_type,...,ai_subject,actual_subject,ai_points_requested,actual_points_requested,ai_draft_email,actual_human_email,human_edit_types,human_edit_summary,final_status,synthetic_internal_note
0,1,HL-2026-549245,Madison Scott / Archie Parker,Joint,WA,Direct digital,Refinance,Investment,Conditional approval,Self-employed income verification,...,Application review - please provide documents,Thanks for your application - a few items to c...,the last 2 years of Notices of Assessment | th...,the latest 2 years of personal and business No...,"Hi Madison and Archie,\n\nThank you for applyi...","Hi Madison and Archie,\n\nThanks for submittin...",removed a generic deadline; tightened the scop...,Human version is more specific about the evide...,Condition cleared,Self-employed income needs standard verificati...
1,2,HL-2025-575763,Max Smith,Single,ACT,Broker-introduced,Top-up,Owner Occupied,Conditional approval,Bonus/overtime verification,...,Home loan application - additional information...,Thanks for your application - a few items to c...,your last 3 payslips showing overtime or bonus...,your 2 most recent payslips showing the overti...,"Hi Max,\n\nThanks for your home loan applicati...","Hi Max,\n\nI've reviewed your application and ...",removed a generic deadline; reordered requests...,Human version improves clarity and sequencing....,Awaiting customer response,Variable income has been included in servicing...
2,3,HL-2026-172574,Alexander Nguyen,Single,NSW,Online refinance,Purchase,Investment,Verification,Dependants and family costs,...,Follow up on your application - dependants and...,Application follow up: dependants and expenses,confirmation of the number of dependants in yo...,confirmation of the number of dependants to be...,"Hi Alexander,\n\nThank you for applying with C...","Hi Alexander,\n\nThanks for your application. ...",made the request more customer-friendly; added...,Human version is more specific about the evide...,Duplicate application closed,Household composition and related costs do not...
3,4,HL-2025-801474,Thomas White,Single,NSW,Direct digital,Refinance,Investment,Verification,Outstanding bank statements,...,Application review - please provide documents,Application follow up: bank statements,the last 3 months of your main transaction acc...,the most recent statements for the main accoun...,"Hi Thomas,\n\nI am writing regarding your appl...","Hi Thomas,\n\nThanks for submitting your appli...",more specific document wording; reduced ambigu...,Human version is more specific about the evide...,Withdrawn by customer,Uploaded statements are incomplete or do not c...
4,5,HL-2025-179046,Amelia Williams,Single,WA,Direct digital,Refinance,Investment,Verification,Residency / visa evidence,...,Application review - please provide documents,Quick follow up on your application - residenc...,a copy of your visa or residency document | co...,a copy of your current visa or residency docum...,"Hi Amelia,\n\nThanks for your home loan applic...","Hi Amelia,\n\nThanks for your application. I h...",added alternative evidence option; added next-...,Human version softens the tone and avoids gene...,Condition cleared,Residency evidence is required for policy veri...


In [31]:
import pandas as pd
import numpy as np

## Use numpy since it's known for speed, vectorisation + numerical operations at scale - faster and reliable data processing + analysis

In [32]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [33]:
df['ai_length'] = df['ai_draft_email'].str.len()
df['human_length'] = df['actual_human_email'].str.len()

df['length_diff'] = df['human_length'] - df['ai_length']

In [34]:
df['ai_sentences'] = df['ai_draft_email'].str.count(r'\.') + 1
df['human_sentences'] = df['actual_human_email'].str.count(r'\.') + 1

In [35]:
def compare_points(ai, human):
    ai_set = set(str(ai).split(','))
    human_set = set(str(human).split(','))

    added = human_set - ai_set
    removed = ai_set - human_set

    return len(added), len(removed)

df[['points_added', 'points_removed']] = df.apply(
    lambda x: pd.Series(compare_points(x['ai_points_requested'], x['actual_points_requested'])),
    axis=1
)

In [41]:
import pandas as pd
from difflib import SequenceMatcher

df['ai_text'] = df['ai_draft_email'].astype(str)
df['human_text'] = df['actual_human_email'].astype(str)

df['ai_char_count'] = df['ai_text'].str.len()
df['human_char_count'] = df['human_text'].str.len()
df['char_count_diff'] = df['human_char_count'] - df['ai_char_count']

df['ai_word_count'] = df['ai_text'].str.split().str.len()
df['human_word_count'] = df['human_text'].str.split().str.len()
df['word_count_diff'] = df['human_word_count'] - df['ai_word_count']

In [42]:
def text_similarity(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

df['text_similarity'] = df.apply(
    lambda x: text_similarity(x['ai_text'], x['human_text']),
    axis=1
)

df['text_variance'] = 1 - df['text_similarity']

In [43]:
def text_similarity(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

df['text_similarity'] = df.apply(
    lambda x: text_similarity(x['ai_text'], x['human_text']),
    axis=1
)

df['text_variance'] = 1 - df['text_similarity']

In [44]:
df['ai_sentence_count'] = df['ai_text'].str.count(r'[.!?]') + 1
df['human_sentence_count'] = df['human_text'].str.count(r'[.!?]') + 1
df['sentence_count_diff'] = df['human_sentence_count'] - df['ai_sentence_count']

In [45]:
df['ai_paragraph_count'] = df['ai_text'].str.split(r'\n\s*\n').str.len()
df['human_paragraph_count'] = df['human_text'].str.split(r'\n\s*\n').str.len()
df['paragraph_count_diff'] = df['human_paragraph_count'] - df['ai_paragraph_count']

In [46]:
def added_removed_words(ai, human):
    ai_words = set(str(ai).lower().split())
    human_words = set(str(human).lower().split())

    added = human_words - ai_words
    removed = ai_words - human_words

    return len(added), len(removed)

df[['words_added_count', 'words_removed_count']] = df.apply(
    lambda x: pd.Series(added_removed_words(x['ai_text'], x['human_text'])),
    axis=1
)

In [47]:
def edit_ratio(a, b):
    return 1 - SequenceMatcher(None, str(a), str(b)).ratio()

df['edit_ratio'] = df.apply(
    lambda x: edit_ratio(x['ai_text'], x['human_text']),
    axis=1
)

In [48]:
df['body_change_score'] = (
    (1 - df['text_similarity']) * 0.5 +
    (df['word_count_diff'].abs() / df[['ai_word_count', 'human_word_count']].max(axis=1).replace(0, 1)) * 0.2 +
    (df['words_added_count'] / df['human_word_count'].replace(0, 1)) * 0.15 +
    (df['words_removed_count'] / df['ai_word_count'].replace(0, 1)) * 0.15
) * 100

In [49]:
df[['text_similarity', 'text_variance', 'word_count_diff', 'words_added_count', 'words_removed_count', 'body_change_score']].describe()

,text_similarity,text_variance,word_count_diff,words_added_count,words_removed_count,body_change_score
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,0.116316,0.883684,19.640000,44.050000,29.450000,58.911642
std,0.065266,0.065266,12.470183,7.411941,4.944898,4.359575
min,0.028470,0.706667,-1.000000,29.000000,18.000000,48.971899
25%,0.061443,0.841162,10.000000,39.000000,26.000000,55.129355
50%,0.093172,0.906828,17.000000,43.000000,29.000000,59.944179
75%,0.158838,0.938557,29.000000,50.000000,33.000000,62.528741
max,0.293333,0.971530,50.000000,65.000000,43.000000,67.195318


In [50]:
df.groupby('scenario_type')[['text_similarity', 'body_change_score']].mean().sort_values('body_change_score', ascending=False)

,text_similarity,body_change_score
scenario_type,,
Debt reduction confirmation,0.095448,63.476733
Living expenses clarification,0.054113,63.271668
Loan term clarification,0.071736,62.279783
Credit card limits / closure,0.083374,61.903856
Refinance payout details,0.122883,60.921816
Outstanding bank statements,0.088145,60.862355
Duplicate application closure,0.100132,60.544646
Undisclosed liability clarification,0.130869,60.509692
Purchase contract clarification,0.098278,60.328545


In [51]:
df.groupby('lender_name')[['text_similarity', 'body_change_score']].mean().sort_values('body_change_score', ascending=False)

,text_similarity,body_change_score
lender_name,,
James Pritchard,0.068945,61.499169
Chloe Haddad,0.099374,61.477908
Sarah Tomic,0.081299,61.159422
Ben Waller,0.079118,60.877789
Olivia Nguyen,0.113488,59.223943
Priya Sharma,0.120701,58.766240
Lachlan Fraser,0.106576,58.311302
Emily Russo,0.126082,57.784925
Daniel Kwan,0.175896,55.388116


In [52]:
## Minimal version

df['ai_text'] = df['ai_draft_email'].astype(str)
df['human_text'] = df['actual_human_email'].astype(str)

def text_similarity(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

def added_removed_words(ai, human):
    ai_words = set(str(ai).lower().split())
    human_words = set(str(human).lower().split())
    return len(human_words - ai_words), len(ai_words - human_words)

df['text_similarity'] = df.apply(
    lambda x: text_similarity(x['ai_text'], x['human_text']),
    axis=1
)

df['text_variance'] = 1 - df['text_similarity']

df['ai_word_count'] = df['ai_text'].str.split().str.len()
df['human_word_count'] = df['human_text'].str.split().str.len()
df['word_count_diff'] = df['human_word_count'] - df['ai_word_count']

df[['words_added_count', 'words_removed_count']] = df.apply(
    lambda x: pd.Series(added_removed_words(x['ai_text'], x['human_text'])),
    axis=1
)

df[['record_id', 'text_similarity', 'text_variance', 'word_count_diff', 'words_added_count', 'words_removed_count']].head()

,record_id,text_similarity,text_variance,word_count_diff,words_added_count,words_removed_count
0,1,0.271583,0.728417,17,43,27
1,2,0.059087,0.940913,5,38,29
2,3,0.095830,0.904170,19,47,29
3,4,0.042088,0.957912,20,52,32
4,5,0.189474,0.810526,4,33,24
